# FIT5201 Assignment 2 - Section 1: Document Clustering
**Student Name**: [Yin Zhang]
**Student ID**: [[Student ID removed]]

## Question 1: Document Clustering with EM Algorithm

### I. Theory (12 marks)

**Mixture model.** Each document $\mathbf{x}_i$ (bag-of-words counts) is generated by a latent cluster $z_i \in \{1,\ldots,K\}$ with prior $\phi_k = P(z_i=k)$ and multinomial parameters $\boldsymbol{\mu}_k$ (word probabilities).

**Complete-data log-likelihood** (observed $\mathbf{x}_i$ and known $z_i=k$):
$$\ell_c(\phi, \boldsymbol{\mu}) = \sum_{i=1}^{N} \sum_{k=1}^{K} \mathbb{1}[z_i=k]\left(\log \phi_k + \sum_{d=1}^{D} x_{id}\log \mu_{kd}\right).$$

**Incomplete-data log-likelihood** (latent $z_i$ marginalized):
$$\ell(\phi, \boldsymbol{\mu}) = \sum_{i=1}^{N} \log \left(\sum_{k=1}^{K} \phi_k \prod_{d=1}^{D} \mu_{kd}^{x_{id}}\right).$$
Direct maximization is hard because $\log\sum_k$ couples parameters and creates local maxima.

**EM (soft).** E-step: responsibilities $\gamma_{ik} = P(z_i=k\mid \mathbf{x}_i) \propto \phi_k \prod_d \mu_{kd}^{x_{id}}$. M-step:
$$\phi_k = \frac{1}{N}\sum_i \gamma_{ik}, \quad \mu_{kd} = \frac{\sum_i \gamma_{ik} x_{id}}{\sum_i \gamma_{ik} \sum_d x_{id}}.$$
Use log-sum-exp for stable computation of $\gamma_{ik}$.

**Hard-EM.** E-step: $z_i = \arg\max_k \log \phi_k + \sum_d x_{id}\log \mu_{kd}$. M-step uses cluster counts instead of $\gamma_{ik}$. This is a local optimum of a lower bound and can be less stable than soft EM.

**Why incomplete MLE is difficult:** the marginal log-likelihood is non-concave with many local maxima; EM monotonically improves a tractable surrogate (ELBO) instead.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

%matplotlib inline

def save_show(path=None):
    if path:
        plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

with open("Dataset_S2_2025/Task2A.txt", "r", encoding="utf-8") as f:
    documents = f.readlines()

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2
)
X = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")

EPS = 1e-10

def logsumexp(x):
    c = np.max(x, axis=1, keepdims=True)
    return c + np.log(np.sum(np.exp(x - c), axis=1, keepdims=True))

def init_mu_from_documents(X, K, random_state):
    rng = np.random.RandomState(random_state)
    idx = rng.choice(X.shape[0], K, replace=False)
    mu = np.zeros((K, X.shape[1]))
    for k, i in enumerate(idx):
        v = X[i].toarray().ravel().astype(float) + 1e-3
        mu[k] = v / v.sum()
    return mu

def log_joint_matrix(X, phi, mu):
    log_phi = np.log(phi + EPS)
    log_mu = np.log(mu + EPS)
    return log_phi + X.dot(log_mu.T)

def mixture_mean_loglik(X, phi, mu):
    lj = log_joint_matrix(X, phi, mu)
    return float(np.mean(logsumexp(lj)))

def soft_em(X, K, max_iter=200, tol=1e-5, random_state=None):
    N, D = X.shape
    phi = np.ones(K) / K
    mu = init_mu_from_documents(X, K, random_state)
    prev_ll = -np.inf

    for it in range(max_iter):
        log_joint = log_joint_matrix(X, phi, mu)
        log_norm = logsumexp(log_joint)
        gamma = np.exp(log_joint - log_norm)

        phi = gamma.mean(axis=0)
        phi = np.maximum(phi, EPS)
        phi /= phi.sum()

        for k in range(K):
            w = gamma[:, k]
            if w.sum() < EPS:
                mu[k] = init_mu_from_documents(X, 1, (random_state or 0) + k + 17)[0]
                continue
            num = X.multiply(w[:, None]).sum(axis=0).A1
            mu[k] = num / (num.sum() + EPS)

        ll = mixture_mean_loglik(X, phi, mu)
        if abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    return phi, mu, gamma, it + 1

def hard_em(X, K, max_iter=200, tol=1e-5, random_state=None):
    N, D = X.shape
    rng = np.random.RandomState(random_state)
    phi = np.ones(K) / K
    mu = init_mu_from_documents(X, K, random_state)
    prev_ll = -np.inf

    for it in range(max_iter):
        log_joint = log_joint_matrix(X, phi, mu)
        z = np.argmax(log_joint, axis=1)

        counts = np.bincount(z, minlength=K)
        phi = np.maximum(counts / N, EPS)
        phi /= phi.sum()

        for k in range(K):
            mask = z == k
            if not mask.any():
                i = rng.randint(0, N)
                v = X[i].toarray().ravel().astype(float) + 1e-3
                mu[k] = v / v.sum()
                continue
            num = X[mask].sum(axis=0).A1
            mu[k] = num / (num.sum() + EPS)

        ll = mixture_mean_loglik(X, phi, mu)
        if abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    return phi, mu, z, it + 1

In [ ]:
# III. Experiments (10 marks)
from IPython.display import Markdown, display

K = 4
num_initializations = 3
soft_results, hard_results = [], []

print("Running experiments with K=4 and 3 random initializations...")

for i in range(num_initializations):
    print(f"\nSoft EM - Initialization {i+1}")
    phi_s, mu_s, gamma_s, n_s = soft_em(X, K, random_state=i)
    ll_s = mixture_mean_loglik(X, phi_s, mu_s)
    soft_results.append({'loglik': ll_s, 'mu': mu_s, 'gamma': gamma_s, 'iters': n_s})

    print(f"Hard EM - Initialization {i+1}")
    phi_h, mu_h, z_h, n_h = hard_em(X, K, random_state=i)
    ll_h = mixture_mean_loglik(X, phi_h, mu_h)
    hard_results.append({'loglik': ll_h, 'mu': mu_h, 'assignments': z_h, 'iters': n_h})

best_soft = max(soft_results, key=lambda x: x['loglik'])
best_hard = max(hard_results, key=lambda x: x['loglik'])

soft_lls = [r['loglik'] for r in soft_results]
hard_lls = [r['loglik'] for r in hard_results]

soft_spread = max(soft_lls) - min(soft_lls)
hard_spread = max(hard_lls) - min(hard_lls)

avg_soft, avg_hard = np.mean(soft_lls), np.mean(hard_lls)

print("\nExperiment Results:")
print(f"Soft EM  — best LL: {best_soft['loglik']:.4f}, avg: {avg_soft:.4f}, iters: {best_soft['iters']}")
print(f"Hard EM  — best LL: {best_hard['loglik']:.4f}, avg: {avg_hard:.4f}, iters: {best_hard['iters']}")

bs = np.argmax(best_soft['gamma'], axis=1)
print(f"Best Soft cluster sizes: {[int((bs==k).sum()) for k in range(K)]}")
print(f"Best Hard cluster sizes: {[int((best_hard['assignments']==k).sum()) for k in range(K)]}")

exp_md = f"""### III. Experiment discussion

- **Soft EM** (3 runs): best log-likelihood = {best_soft['loglik']:.4f}, average = {avg_soft:.4f}, range = {soft_spread:.4f}, iterations = {best_soft['iters']}.
- **Hard EM** (3 runs): best log-likelihood = {best_hard['loglik']:.4f}, average = {avg_hard:.4f}, range = {hard_spread:.4f}, iterations = {best_hard['iters']}.

**Initialization sensitivity:** """

if soft_spread > 1e-3:
    exp_md += f"Soft EM varies across seeds, with a spread of {soft_spread:.4f}. "
else:
    exp_md += f"Soft EM is similar across seeds, with a spread of {soft_spread:.4f}. "

if hard_spread > 1e-3:
    exp_md += f"Hard EM also shows sensitivity to initialization, with a spread of {hard_spread:.4f}."
else:
    exp_md += f"Hard EM is relatively stable across seeds, with a spread of {hard_spread:.4f}."

exp_md += (
    "\n\nThe likelihood is computed using **x·log(μ+ε)** under the multinomial mixture model. "
    "Both Soft EM and Hard EM are reported using the same mean incomplete-data log-likelihood, "
    "so their final likelihood values can be compared more consistently."
)

display(Markdown(exp_md))

In [ ]:
# IV. Interpretation and visualization (8 marks)

from sklearn.preprocessing import normalize
from IPython.display import Markdown, display

def plot_clusters(X, assignments, title, filename):
    """
    Plot document clusters using PCA.

    The EM model is trained on the original bag-of-words count matrix X.
    For visualization only, the document vectors are L2-normalised before PCA
    to reduce the effect of document length and extreme outliers.
    """
    X_norm = normalize(X, norm='l2', axis=1)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_norm.toarray())

    plt.figure(figsize=(10, 6))

    for k in range(K):
        m = assignments == k
        if not np.any(m):
            continue

        plt.scatter(
            X_pca[m, 0],
            X_pca[m, 1],
            label=f'Cluster {k+1} (n={m.sum()})',
            alpha=0.7,
            s=20
        )

    plt.title(title)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.legend()
    save_show(filename)


def get_top_words(mu, k, n=10):
    idx = np.argsort(mu[k])[::-1][:n]
    return [vocab[i] for i in idx]


print("Visualizing clustering results...")

soft_assign = np.argmax(best_soft['gamma'], axis=1)

plot_clusters(
    X,
    soft_assign,
    'Soft EM Clustering (PCA Visualization)',
    'soft_em_pca.png'
)

plot_clusters(
    X,
    best_hard['assignments'],
    'Hard EM Clustering (PCA Visualization)',
    'hard_em_pca.png'
)

print("\nTop 10 words per cluster:")

for name, mu in [('Soft EM', best_soft['mu']), ('Hard EM', best_hard['mu'])]:
    print(name + ':')
    for k in range(K):
        print(f"  Cluster {k+1}: {', '.join(get_top_words(mu, k))}")


# Cluster statistics
max_resp = np.max(best_soft['gamma'], axis=1)
ambig_doc = int(np.argmin(max_resp))
ambig_prob = float(max_resp.min())

soft_sizes = [int(np.sum(soft_assign == k)) for k in range(K)]
hard_sizes = [int(np.sum(best_hard['assignments'] == k)) for k in range(K)]

soft_word_lists = [tuple(get_top_words(best_soft['mu'], k)) for k in range(K)]
hard_word_lists = [tuple(get_top_words(best_hard['mu'], k)) for k in range(K)]

soft_distinct = len(set(soft_word_lists))
hard_distinct = len(set(hard_word_lists))

soft_entropy = float(
    -np.sum(best_soft['gamma'] * np.log(best_soft['gamma'] + EPS)) 
    / len(best_soft['gamma'])
)

n_empty_soft = sum(1 for s in soft_sizes if s == 0)
n_empty_hard = sum(1 for s in hard_sizes if s == 0)

if ambig_prob < 0.8:
    ambig_comment = (
        "This document is relatively ambiguous because its highest cluster "
        "responsibility is not close to 1."
    )
else:
    ambig_comment = (
        "This is the least confident document under Soft EM, although the "
        "assignment is still relatively confident."
    )


interp_md = f"""### IV. Interpretation

1. **Cluster sizes (argmax γ / hard labels):** Soft EM {soft_sizes}; Hard EM {hard_sizes}. Empty clusters: soft {n_empty_soft}, hard {n_empty_hard}.

2. **Lexical separation:** {soft_distinct} distinct Soft-EM top-word profiles vs {hard_distinct} for Hard EM (K={K}). Mean max responsibility = {max_resp.mean():.4f}; entropy of γ = {soft_entropy:.4f}.

3. **Quality:** Hard EM produces four non-empty clusters and the top-word lists suggest some topic separation. However, several clusters still share generic newsgroup words such as `edu`, `sci`, `com`, and `writes`, so the interpretation should be treated cautiously. The PCA plots are based on L2-normalised document-term vectors to reduce the influence of document length and outliers. This normalisation is used only for visualisation and does not change the EM training results.

4. **Ambiguous document (Soft EM):** doc **{ambig_doc}** has the lowest max γ = **{ambig_prob:.4f}**. {ambig_comment}

From the top words, Cluster 1 appears to represent space-related documents, Cluster 2 is related to computing or encryption, Cluster 3 contains medical or science terms, and Cluster 4 appears to contain electronics or general technical discussion.

| Cluster | Soft EM top-5 words | Hard EM top-5 words |
|---------|---------------------|---------------------|
"""

for k in range(K):
    interp_md += (
        f"| {k+1} | "
        f"{', '.join(get_top_words(best_soft['mu'], k, 5))} | "
        f"{', '.join(get_top_words(best_hard['mu'], k, 5))} |\n"
    )

display(Markdown(interp_md))